In [4]:
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import date, datetime, timedelta
import random
import time
import os
import duckdb
DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"

con = duckdb.connect(DB_PATH)
print("Connected:", DB_PATH)
# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb


In [ ]:

# con.close()
# print("DuckDB connection closed.")

DuckDB connection closed.


In [ ]:
# =========================================
# SEASON DATE RANGES
# =========================================
season_start_dates = {
    "20132014": "2013-10-01",
    "20142015": "2014-10-08",
    "20152016": "2015-10-07",
    "20162017": "2016-10-12",
    "20172018": "2017-10-04",
    "20182019": "2018-10-04",
    "20192020": "2019-10-02",
    "20202021": "2021-01-13",
    "20212022": "2021-10-12",
    "20222023": "2022-10-11",
    "20232024": "2023-10-11",
    "20242025": "2024-10-04",
    "20252026": "2025-10-07"
}

season_end_dates = {
    "20132014": "2014-04-13",
    "20142015": "2015-04-11",
    "20152016": "2016-04-10",
    "20162017": "2017-04-09",
    "20172018": "2018-04-08",
    "20182019": "2019-04-06",
    "20192020": "2020-03-12",
    "20202021": "2021-05-19",
    "20212022": "2022-04-29",
    "20222023": "2023-04-13",
    "20232024": "2024-04-18",
    "20242025": "2025-04-17",
    "20252026": "2025-12-05"
}

# =========================================
# SAFE REQUEST
# =========================================
def safe_get(url, retries=5):
    wait = 1.0

    for _ in range(retries):
        r = requests.get(url)

        if r.status_code == 200:
            time.sleep(0.15 + random.uniform(0, 0.15))
            return r

        if r.status_code == 429:
            print(f"[429] Rate limited — waiting {wait:.1f}s…")
            time.sleep(wait)
            wait *= 2
            continue

        print(f"[{r.status_code}] Error fetching {url}")
        return None

    print("❌ Max retries reached:", url)
    return None


# =========================================
# DAILY SNAPSHOT SCRAPER
# =========================================
def get_daily_standings_snapshots(season_start_dates, season_end_dates):
    base_url = "https://api-web.nhle.com/v1/standings/"
    all_rows = []

    for season_id, start_str in season_start_dates.items():
        end_str = season_end_dates[season_id]

        start_date = pd.to_datetime(start_str)
        end_date = pd.to_datetime(end_str)

        print(f"\n==============================")
        print(f"📅 Processing season {season_id}")
        print(f"   From {start_str} → {end_str}")
        print("==============================\n")

        for dt in pd.date_range(start_date, end_date, freq="D"):
            date_str = dt.strftime("%Y-%m-%d")
            url = f"{base_url}{date_str}"

            r = safe_get(url)
            if r is None:
                print(f"⚠ Skipping {date_str}")
                continue

            data = r.json().get("standings", [])
            if not data:
                print(f"⚠ No standings for {date_str}")
                continue

            for team in data:
                all_rows.append({
                    "game_date": date_str,
                    "date": date_str,
                    "season_id": season_id,

                    "tri_code": team.get("teamAbbrev", {}).get("default", None),
                    "team_name": team.get("teamName", {}).get("default",None),
                    "games_played": team.get("gamesPlayed",0),
                    "road_gp": team.get("roadGamesPlayed",0),

                    "division": team.get("divisionName"),
                    "division_rank": team.get("divisionSequence", 0),
                    "division_home_rank":team.get("divisionHomeSequence", 0),
                    "division_road_rank":team.get("divisionRoadSequence", 0),
                    "division_last_10_rank":team.get("divisionL10Sequence", 0),
                    
                    "conference": team.get("conferenceName"),
                    "conference_rank": team.get("conferenceSequence", 0),
                    "conference_home_rank": team.get("conferenceHomeSequence", 0),
                    "conference_road_rank": team.get("conferenceRoadSequence", 0),
                    "conference_last_10_rank": team.get("conferenceL10Sequence", 0),
                    
                    
                    "league_rank": team.get("leagueSequence", 0),
                    "league_home_rank": team.get("leagueHomeSequence", 0),
                    "league_road_rank": team.get("leagueRoadSequence", 0),
                    "league_last_10_rank": team.get("leagueL10Sequence", 0),
                    "wildcard_rank": team.get("wildcardSequence", 0),

                    
                    "wins": team.get("wins", 0),
                    "wins_pctg": team.get("winPctg", 0.0),
                    "rw":team.get("regulationWins",0),
                    "losses": team.get("losses", 0),
                    "ot_losses": team.get("otLosses", 0),
                    "rotw": team.get('regulationPlusOtWins',0),
                    "rotw_pctg": team.get("regulationPlusOtWinPctg", 0),
                    "road_losses": team.get("roadLosses", 0),
                    "road_ot_losses": team.get("roadOtLosses", 0),
                    "road_rotw": team.get("roadRegulationPlusOtWins", 0),
                    "road_rw": team.get("roadRegulationWins",0),
                    
                    
                    
                    
                    "points": team.get("points", 0),
                    "points_pct": team.get("pointPctg", 0),
                    "home_points": team.get("homePoints",0),
                    "road_points": team.get("roadPoints", 0),

                    "goals_for": team.get("goalFor",0),
                    "gf_pctg": team.get("goalsForPctg", 0.0),
                    "home_gf": team.get("homeGoalsFor", 0),
                    "road_gf": team.get("roadGoalsFor", 0),
                    
                    
                    "goals_against": team.get("goalAgainst",0),
                    "home_ga": team.get("homeGoalsAgainst", 0),
                    "road_ga": team.get("roadGoalsAgainst", 0),
                    
                    
                    "goal_diff": team.get("goalDifferential", 0),
                    "goal_diff_pctg":team.get("goalDifferentialPctg",0.0),
                    "home_goal_diff": team.get("homeGoalDifferential", 0),
                    "road_goal_diff": team.get("roadGoalDifferential", 0),
                    
                    
                    "streak": team.get("streakCode"),
                    "streak_count": team.get("streakCount"),


                    "last_10_gp": team.get("l10GamesPlayed", 0),
                    "last_10_wins": team.get("l10Wins", 0),
                    "last_10_goal_diff": team.get("l10GoalDifferential", 0),
                    "last_10_ga": team.get("l10GoalsAgainst", 0),
                    "last_10_gf": team.get("l10GoalsFor", 0),
                    "last_10_losses": team.get("l10Losses", 0),
                    "last_10_otl": team.get("l10OtLosses", 0),
                    "last_10_points": team.get("l10Points", 0),
                    "last_10_rotw": team.get("l10RegulationPlusOtWins", 0),
                    "last_10_rw": team.get("l10RegulationWins", 0),
                    
                    
                    "home_gp": team.get("homeGamesPlayed", 0),
                    "home_wins": team.get("homeWins", 0),
                    "home_losses": team.get("homeLosses", 0),
                    "home_otlosses": team.get("homeOtLosses", 0),
                    "home_rotw": team.get("homeRegulationPlusOtWins", 0),
                    "home_rw": team.get("homeRegulationWins", 0),
                    "road_wins": team.get("roadWins", 0),
                    
                    "so_losses": team.get("shootoutLosses", 0),
                    "so_wins": team.get("shootoutWins", 0),

                    "in_playoff_spot": 
                        1 if (
                            (team.get("divisionRank") is not None and team.get("divisionRank") <= 3) or
                            (team.get("wildCardRank") is not None and team.get("wildCardRank") <= 2)
                        ) else 0
                })

        print(f"✔ Completed season {season_id}")

    df = pd.DataFrame(all_rows)
    print(f"\n📦 Total snapshot rows collected: {len(df):,}")
    return df


# =========================================
# RUN SCRAPER
# =========================================
standings_df = get_daily_standings_snapshots(season_start_dates, season_end_dates)




📅 Processing season 20132014
   From 2013-10-01 → 2014-04-13

✔ Completed season 20132014

📅 Processing season 20142015
   From 2014-10-08 → 2015-04-11

✔ Completed season 20142015

📅 Processing season 20152016
   From 2015-10-07 → 2016-04-10

✔ Completed season 20152016

📅 Processing season 20162017
   From 2016-10-12 → 2017-04-09

✔ Completed season 20162017

📅 Processing season 20172018
   From 2017-10-04 → 2018-04-08

✔ Completed season 20172018

📅 Processing season 20182019
   From 2018-10-04 → 2019-04-06

✔ Completed season 20182019

📅 Processing season 20192020
   From 2019-10-02 → 2020-03-12

⚠ No standings for 2020-03-12
✔ Completed season 20192020

📅 Processing season 20202021
   From 2021-01-13 → 2021-05-19

✔ Completed season 20202021

📅 Processing season 20212022
   From 2021-10-12 → 2022-04-29

✔ Completed season 20212022

📅 Processing season 20222023
   From 2022-10-11 → 2023-04-13

✔ Completed season 20222023

📅 Processing season 20232024
   From 2023-10-11 → 2024-04-1

In [8]:
standings_df['games_remaining'] = 82-standings_df['games_played'].fillna(0)
standings_df['ppg'] = standings_df['points']/standings_df['games_played']
standings_df["projected_points"] = (
    standings_df["points"]
    + (
        standings_df["ppg"].fillna(0)
        * standings_df["games_remaining"]
    )
).round().astype("Int64")

standings_df['team_id'] = np.nan
standings_df.head()

,game_date,date,season_id,tri_code,team_name,games_played,road_gp,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,wins,wins_pctg,rw,losses,ot_losses,rotw,rotw_pctg,road_losses,road_ot_losses,road_rotw,road_rw,points,points_pct,home_points,road_points,goals_for,gf_pctg,home_gf,road_gf,goals_against,home_ga,road_ga,goal_diff,goal_diff_pctg,home_goal_diff,road_goal_diff,streak,streak_count,last_10_gp,last_10_wins,last_10_goal_diff,last_10_ga,last_10_gf,last_10_losses,last_10_otl,last_10_points,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,road_wins,so_losses,so_wins,in_playoff_spot,games_remaining,ppg,projected_points,team_id
0,2013-10-01,2013-10-01,20132014,CHI,Chicago Blackhawks,1,0,Central,1,1,2,1,Western,1,1,4,1,1,1,8,1,0,1,1.0,1,0,0,1,1.0,0,0,0,0,2,1.0,2,0,6,6.0,6,0,4,4,0,2,2.0,2,0,W,1.0,1,1,2,4,6,0,0,2,1,1,1,1,0,0,1,1,0,0,0,0,81,2.0,164,NaN
1,2013-10-01,2013-10-01,20132014,TOR,Toronto Maple Leafs,1,1,Atlantic,1,7,1,1,Eastern,1,14,1,1,2,25,1,2,0,1,1.0,1,0,0,1,1.0,0,0,1,1,2,1.0,0,2,4,4.0,0,4,3,0,3,1,1.0,0,1,W,1.0,1,1,1,3,4,0,0,2,1,1,0,0,0,0,0,0,1,0,0,0,81,2.0,164,NaN
2,2013-10-01,2013-10-01,20132014,WPG,Winnipeg Jets,1,1,Central,2,7,1,2,Western,2,13,1,2,3,28,2,3,0,1,1.0,1,0,0,1,1.0,0,0,1,1,2,1.0,0,2,5,5.0,0,5,4,0,4,1,1.0,0,1,W,1.0,1,1,1,4,5,0,0,2,1,1,0,0,0,0,0,0,1,0,0,0,81,2.0,164,NaN
3,2013-10-01,2013-10-01,20132014,ANA,Anaheim Ducks,0,0,Pacific,1,1,1,1,Western,3,2,2,3,4,2,3,4,0,0,0.0,0,0,0,0,0.0,0,0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0,0.0,0,0,None,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,82,NaN,0,NaN
4,2013-10-01,2013-10-01,20132014,BOS,Boston Bruins,0,0,Atlantic,2,1,2,2,Eastern,2,1,2,2,5,3,4,5,0,0,0.0,0,0,0,0,0.0,0,0,0,0,0,0.0,0,0,0,0.0,0,0,0,0,0,0,0.0,0,0,None,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,82,NaN,0,NaN


In [ ]:
# # con.execute("DROP TABLE IF EXISTS all_standings_daily")
# # con.execute("""
# # CREATE TABLE IF NOT EXISTS all_standings_daily AS
# # SELECT * FROM standings_df LIMIT 0
# # """)

# # con.execute("""
# # INSERT INTO all_standings_daily
# # SELECT * FROM standings_df
# # """)

# # print("✔ data inserted into DuckDB.")



con.execute('select * from all_standings_daily').fetch_df().tail()

,game_date,date,season_id,tri_code,team_name,games_played,road_gp,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,wins,wins_pctg,rw,losses,ot_losses,rotw,rotw_pctg,road_losses,road_ot_losses,road_rotw,road_rw,points,points_pct,home_points,road_points,goals_for,gf_pctg,home_gf,road_gf,goals_against,home_ga,road_ga,goal_diff,goal_diff_pctg,home_goal_diff,road_goal_diff,streak,streak_count,last_10_gp,last_10_wins,last_10_goal_diff,last_10_ga,last_10_gf,last_10_losses,last_10_otl,last_10_points,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,road_wins,so_losses,so_wins,in_playoff_spot,games_remaining,ppg,projected_points,team_id
69550,2025-12-05,2025-12-05,20252026,BUF,Buffalo Sabres,28,12,Atlantic,8,2,8,3,Eastern,16,3,16,10,28,7,32,17,10,11,0.392857,8,13,4,9,0.321429,8,2,1,0,26,0.464286,20,6,84,3.000000,54,30,96,45,51,-12,-0.428571,9,-21,L,2.0,10,5,1,32,33,5,0,10,4,4,16,9,5,2,8,8,2,0,2,0,54,0.928571,76,7.0
69551,2025-12-05,2025-12-05,20252026,STL,St. Louis Blues,28,13,Central,7,7,7,8,Western,13,12,14,15,29,27,27,30,7,9,0.321429,9,12,7,9,0.321429,6,3,4,4,25,0.446429,14,11,71,2.535714,37,34,100,57,43,-29,-1.035714,-20,-9,L,2.0,10,3,-10,29,19,4,3,9,3,3,15,5,6,4,5,5,4,2,0,0,54,0.892857,73,19.0
69552,2025-12-05,2025-12-05,20252026,NSH,Nashville Predators,27,11,Central,8,8,8,5,Western,14,13,15,9,30,28,28,20,8,10,0.370370,7,13,4,9,0.333333,5,2,4,3,24,0.444444,14,10,71,2.629630,42,29,94,55,39,-23,-0.851852,-13,-10,W,2.0,10,5,-8,35,27,5,0,10,5,3,16,6,8,2,5,4,4,0,1,0,55,0.888889,73,18.0
69553,2025-12-05,2025-12-05,20252026,CGY,Calgary Flames,29,18,Pacific,7,6,7,2,Western,15,14,13,5,31,29,26,9,9,10,0.344828,8,15,4,8,0.275862,11,2,4,4,24,0.413793,12,12,70,2.413793,28,42,88,25,63,-18,-0.620690,3,-21,W,1.0,10,5,0,30,30,3,2,12,4,4,11,5,4,2,4,4,5,2,2,0,53,0.827586,68,20.0
69554,2025-12-05,2025-12-05,20252026,VAN,Vancouver Canucks,28,16,Pacific,8,8,3,8,Western,16,16,5,16,32,31,14,32,10,10,0.357143,6,15,3,8,0.285714,7,2,5,4,23,0.410714,7,16,81,2.892857,32,49,102,46,56,-21,-0.750000,-14,-7,L,2.0,10,2,-11,39,28,6,2,6,2,2,12,3,8,1,3,2,7,0,2,0,54,0.821429,67,23.0


In [ ]:
# con.execute("""ALTER TABLE all_standings_daily
# ALTER COLUMN team_id SET DATA TYPE BIGINT;""")


# con.execute("DESCRIBE all_standings_daily").fetchdf()

,column_name,column_type,null,key,default,extra
0,game_date,DATE,YES,None,None,None
1,date,DATE,YES,None,None,None
2,season_id,BIGINT,YES,None,None,None
3,tri_code,VARCHAR,YES,None,None,None
4,team_name,VARCHAR,YES,None,None,None
5,games_played,BIGINT,YES,None,None,None
6,road_gp,BIGINT,YES,None,None,None
7,division,VARCHAR,YES,None,None,None
8,division_rank,BIGINT,YES,None,None,None
9,division_home_rank,BIGINT,YES,None,None,None


In [10]:

# # Create table on first run
# con.execute("""
# CREATE TABLE IF NOT EXISTS all_teams AS
# SELECT * FROM t
# """)

con.execute('select * from all_teams').df().tail()



,team_id,tri_code
30,59,UTA
31,23,VAN
32,54,VGK
33,52,WPG
34,15,WSH


In [20]:
#20132014 last season of PHX --> ARI
#20252026 UTA is 68, previous version is 2425

con.execute("""UPDATE all_standings_daily s
SET team_id = t.team_id
FROM all_teams t
WHERE s.tri_code = t.tri_code
  AND s.tri_code != 'UTA';
     """)

con.execute("""UPDATE all_standings_daily
SET team_id = 
    CASE
        WHEN season_id = 20242025 THEN 59
        WHEN season_id >= 20252026 THEN 68
        ELSE team_id
    END
WHERE tri_code = 'UTA';
     """)
con.execute("""SELECT season_id, tri_code, team_id
FROM all_standings_daily
WHERE tri_code = 'UTA'
ORDER BY season_id;
""")

In [ ]:


con.execute('select * from all_standings_daily').df()

,game_date,date,season_id,tri_code,team_name,games_played,road_gp,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,wins,wins_pctg,rw,losses,ot_losses,rotw,rotw_pctg,road_losses,road_ot_losses,road_rotw,road_rw,points,points_pct,home_points,road_points,goals_for,gf_pctg,home_gf,road_gf,goals_against,home_ga,road_ga,goal_diff,goal_diff_pctg,home_goal_diff,road_goal_diff,streak,streak_count,last_10_gp,last_10_wins,last_10_goal_diff,last_10_ga,last_10_gf,last_10_losses,last_10_otl,last_10_points,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,road_wins,so_losses,so_wins,in_playoff_spot,games_remaining,ppg,projected_points,team_id
69550,2025-12-05,2025-12-05,20252026,BUF,Buffalo Sabres,28,12,Atlantic,8,2,8,3,Eastern,16,3,16,10,28,7,32,17,10,11,0.392857,8,13,4,9,0.321429,8,2,1,0,26,0.464286,20,6,84,3.000000,54,30,96,45,51,-12,-0.428571,9,-21,L,2.0,10,5,1,32,33,5,0,10,4,4,16,9,5,2,8,8,2,0,2,0,54,0.928571,76,7
69551,2025-12-05,2025-12-05,20252026,STL,St. Louis Blues,28,13,Central,7,7,7,8,Western,13,12,14,15,29,27,27,30,7,9,0.321429,9,12,7,9,0.321429,6,3,4,4,25,0.446429,14,11,71,2.535714,37,34,100,57,43,-29,-1.035714,-20,-9,L,2.0,10,3,-10,29,19,4,3,9,3,3,15,5,6,4,5,5,4,2,0,0,54,0.892857,73,19
69552,2025-12-05,2025-12-05,20252026,NSH,Nashville Predators,27,11,Central,8,8,8,5,Western,14,13,15,9,30,28,28,20,8,10,0.370370,7,13,4,9,0.333333,5,2,4,3,24,0.444444,14,10,71,2.629630,42,29,94,55,39,-23,-0.851852,-13,-10,W,2.0,10,5,-8,35,27,5,0,10,5,3,16,6,8,2,5,4,4,0,1,0,55,0.888889,73,18
69553,2025-12-05,2025-12-05,20252026,CGY,Calgary Flames,29,18,Pacific,7,6,7,2,Western,15,14,13,5,31,29,26,9,9,10,0.344828,8,15,4,8,0.275862,11,2,4,4,24,0.413793,12,12,70,2.413793,28,42,88,25,63,-18,-0.620690,3,-21,W,1.0,10,5,0,30,30,3,2,12,4,4,11,5,4,2,4,4,5,2,2,0,53,0.827586,68,20
69554,2025-12-05,2025-12-05,20252026,VAN,Vancouver Canucks,28,16,Pacific,8,8,3,8,Western,16,16,5,16,32,31,14,32,10,10,0.357143,6,15,3,8,0.285714,7,2,5,4,23,0.410714,7,16,81,2.892857,32,49,102,46,56,-21,-0.750000,-14,-7,L,2.0,10,2,-11,39,28,6,2,6,2,2,12,3,8,1,3,2,7,0,2,0,54,0.821429,67,23
